In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore", message="`sparse` was renamed")

from sklearn.experimental import enable_halving_search_cv  
from sklearn.model_selection import train_test_split, HalvingGridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [2]:
DATA_PATH = "/kaggle/input/cybersecurity-incident-dataset/cybersecurity synthesized data.csv"
TARGET_COL = "outcome"
DROP_COLS = ["timestamp", "attacker_ip", "target_ip"]
OUT_DIR = "artifacts"
os.makedirs(OUT_DIR, exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.2

In [3]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

In [4]:
df["data_per_min"] = df["data_compromised_GB"] / (df["attack_duration_min"] + 1)
df["response_efficiency"] = df["response_time_min"] / (df["attack_duration_min"] + 1)
df["severity_ratio"] = df["attack_severity"] / (df["response_time_min"] + 1)
df["is_long_attack"] = (df["attack_duration_min"] > 300).astype(int)

df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

In [5]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

In [6]:
le_target = LabelEncoder()
y = le_target.fit_transform(y)
joblib.dump(le_target, os.path.join(OUT_DIR, "label_encoder_target.pkl"))
print("\nLabel Encoding for target variable:")
for cls, enc in zip(le_target.classes_, le_target.transform(le_target.classes_)):
    print(f"{cls} --> {enc}")


Label Encoding for target variable:
Failure --> 0
Success --> 1


In [7]:
le_attack = LabelEncoder()
X["attack_type"] = le_attack.fit_transform(X["attack_type"].astype(str))
joblib.dump(le_attack, os.path.join(OUT_DIR, "label_encoder_attack_type.pkl"))

print("\nLabel Encoding for 'attack_type':")
for k, v in zip(le_attack.classes_, le_attack.transform(le_attack.classes_)):
    print(f"{k} --> {v}")


Label Encoding for 'attack_type':
Brute Force --> 0
Cross-Site Scripting --> 1
DDoS --> 2
Malware --> 3
Phishing --> 4
Ransomware --> 5
SQL Injection --> 6
Zero-Day Exploit --> 7


In [8]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

In [9]:
high_card_cols = [c for c in ["user_role", "security_tools_used", "mitigation_method"] if c in categorical_cols]
low_card_cols = [c for c in categorical_cols if c not in high_card_cols]

le_high = {}
for col in high_card_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_high[col] = le
joblib.dump(le_high, os.path.join(OUT_DIR, "label_encoders_high_cardinality.pkl"))

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

In [10]:
onehot = OneHotEncoder(handle_unknown="ignore", sparse=True)
scaler = StandardScaler(with_mean=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("onehot", onehot, low_card_cols),
        ("scale", scaler, numeric_cols)
    ],
    remainder="drop",
    sparse_threshold=0.3
)

In [11]:
preprocessor

ColumnTransformer(transformers=[('onehot',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse=True),
                                 ['target_system', 'location', 'industry']),
                                ('scale', StandardScaler(with_mean=False),
                                 ['attack_type', 'data_compromised_GB',
                                  'attack_duration_min', 'security_tools_used',
                                  'user_role', 'attack_severity',
                                  'response_time_min', 'mitigation_method',
                                  'data_per_min', 'response_efficiency',
                                  'severity_ratio', 'is_long_attack'])])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

In [13]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",    
    device="cuda",        
    random_state=RANDOM_STATE,
    max_depth=8,
    n_estimators=600,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.8,
    gamma=1,
    reg_lambda=1
)

pipeline = Pipeline([
    ("preproc", preprocessor),
    ("clf", xgb)
])

In [14]:
param_grid = {
    "clf__max_depth": [6, 8, 10],
    "clf__learning_rate": [0.03, 0.05],
    "clf__n_estimators": [400, 600, 800],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0],
    "clf__gamma": [0, 1]
}

In [15]:
print("\n  Running GPU-accelerated HalvingGridSearchCV...")
search = HalvingGridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    factor=3,
    cv=3,
    n_jobs=-1,
    verbose=2
)

search.fit(X_train, y_train)
print("\n Best Parameters:", search.best_params_)

best_model = search.best_estimator_


  Running GPU-accelerated HalvingGridSearchCV...
n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 5
min_resources_: 987
max_resources_: 80000
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 144
n_resources: 987
Fitting 3 folds for each of 144 candidates, totalling 432 fits


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_outp

[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=400, clf__subsample=0.8; total time=   8.1s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=400, clf__subsample=1.0; total time=   7.3s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=600, clf__subsample=1.0; total time=  10.7s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=800, clf__subsample=0.8; total time=  14.8s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=800, clf__subsample=1.0; total time=  14.5s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=8, clf__n_estimators=400, clf__subsample=1.0; total time=   8.8s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=400, clf__subsample=1.0; total time=   7.7s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=400, clf__subsample=1.0; total time=   6.2s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=600, clf__subsample=0.8; total time=  11.1s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=800, clf__subsample=0.8; total time=  14.8s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=800, clf__subsample=1.0; total time=  14.3s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=8, clf__n_estimators=400, clf__subsample=1.0; total time=   8.8s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0

...


[CV] END clf__colsample_bytree=1.0, clf__gamma=0, clf__learning_rate=0.05, clf__max_depth=8, clf__n_estimators=800, clf__subsample=1.0; total time=  18.6s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.05, clf__max_depth=10, clf__n_estimators=400, clf__subsample=0.8; total time=  11.8s
[CV] END clf__colsample_bytree=0.8, clf__gamma=1, clf__learning_rate=0.03, clf__max_depth=8, clf__n_estimators=800, clf__subsample=0.8; total time=  12.1s
[CV] END clf__colsample_bytree=0.8, clf__gamma=1, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=800, clf__subsample=0.8; total time=  11.3s
[CV] END clf__colsample_bytree=1.0, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=400, clf__subsample=0.8; total time=   7.6s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.05, clf__max_depth=8, clf__n_estimators=800, clf__subsample=1.0; total time=  18.5s
[CV] END clf__colsample_bytree=0.8, clf__gamma=1, clf__learning_rate

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(



[CV] END clf__colsample_bytree=1.0, clf__gamma=1, clf__learning_rate=0.05, clf__max_depth=10, clf__n_estimators=800, clf__subsample=1.0; total time=   4.7s
[CV] END clf__colsample_bytree=1.0, clf__gamma=0, clf__learning_rate=0.05, clf__max_depth=8, clf__n_estimators=800, clf__subsample=1.0; total time=  18.6s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.05, clf__max_depth=10, clf__n_estimators=800, clf__subsample=0.8; total time=  22.5s
[CV] END clf__colsample_bytree=0.8, clf__gamma=1, clf__learning_rate=0.03, clf__max_depth=8, clf__n_estimators=800, clf__subsample=0.8; total time=  11.7s
[CV] END clf__colsample_bytree=1.0, clf__gamma=0, clf__learning_rate=0.03, clf__max_depth=6, clf__n_estimators=400, clf__subsample=0.8; total time=   7.5s
[CV] END clf__colsample_bytree=0.8, clf__gamma=0, clf__learning_rate=0.05, clf__max_depth=8, clf__n_estimators=800, clf__subsample=1.0; total time=  18.5s
[CV] END clf__colsample_bytree=1.0, clf__gamma=0, clf__learning_rat

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_outp


 Best Parameters: {'clf__colsample_bytree': 0.8, 'clf__gamma': 0, 'clf__learning_rate': 0.03, 'clf__max_depth': 10, 'clf__n_estimators': 600, 'clf__subsample': 0.8}


In [16]:
y_pred = best_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n Final Test Accuracy: {acc * 100:.2f}%")


 Final Test Accuracy: 49.86%


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [01:23:36] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


In [17]:
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))


Classification Report:
               precision    recall  f1-score   support

           0     0.4983    0.5005    0.4994      9994
           1     0.4989    0.4967    0.4978     10006

    accuracy                         0.4986     20000
   macro avg     0.4986    0.4986    0.4986     20000
weighted avg     0.4986    0.4986    0.4986     20000



In [18]:
joblib.dump(best_model, os.path.join(OUT_DIR, "xgb_cybersecurity_model_gpu.pkl"))
joblib.dump(best_model.named_steps["preproc"], os.path.join(OUT_DIR, "preprocessor.pkl"))

print(f"\n Artifacts saved in '{OUT_DIR}'")
for f in os.listdir(OUT_DIR):
    print("-", os.path.join(OUT_DIR, f))


 Artifacts saved in 'artifacts'
- artifacts/label_encoders_high_cardinality.pkl
- artifacts/label_encoder_target.pkl
- artifacts/preprocessor.pkl
- artifacts/label_encoder_attack_type.pkl
- artifacts/xgb_cybersecurity_model_gpu.pkl


In [19]:
print(df['outcome'].value_counts(normalize=True))

df['outcome_encoded'] = (df['outcome'] == 'Success').astype(int)
corrs = df.corr(numeric_only=True)['outcome_encoded'].sort_values(ascending=False)
print(corrs.head(10))

outcome
Success    0.5003
Failure    0.4997
Name: proportion, dtype: float64
outcome_encoded        1.000000
attack_severity        0.003119
data_compromised_GB    0.002057
severity_ratio         0.001721
attack_duration_min    0.000927
data_per_min           0.000730
response_time_min     -0.002200
response_efficiency   -0.002969
is_long_attack              NaN
Name: outcome_encoded, dtype: float64


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [20]:
!python --version

Python 3.11.13


In [21]:
!pip list

Package                               Version             Editable project location
------------------------------------- ------------------- -------------------------
absl-py                               1.4.0
absolufy-imports                      0.3.1
accelerate                            1.9.0
aiofiles                              22.1.0
aiohappyeyeballs                      2.6.1
aiohttp                               3.12.15
aiosignal                             1.4.0
aiosqlite                             0.21.0
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.11.2
alembic                               1.16.5
altair                                5.5.0
annotated-types                       0.7.0
annoy                                 1.17.3
ansicolors                            1.1.8
antlr4-python3-runtime                4.9.3
anyio                           